**1. Penelaahan Data**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("ruchikakumbhar/placement-prediction-dataset")

print("Path to dataset files:", path)
data = pd.read_csv(os.path.join(path, "placementdata.csv"))
data.head()
#menampilkan dimensi dataset
data.shape

data.info()
#Mengidentifikasi variabel numerik dan kategorik
numerik = data.select_dtypes(include=['int64', 'float64'])
numerik = numerik.drop(columns=['StudentID'])
kategorik = data.select_dtypes(include=['object'])

print("Variabel Numerik:")
print(numerik.columns.tolist())

print("\nVariabel Kategorik:")
print(kategorik.columns.tolist())

print("\nVariabel Identitas:")
print(['StudentID'])

In [ ]:
# Menghitung statistik deskriptif variabel numerik
numerik = data.select_dtypes(include=['int64', 'float64'])

# Mengeluarkan StudentID
numerik = numerik.drop(columns=['StudentID'])

numerik.describe()

In [ ]:
data.describe(include='object')
#Memeriksa distribusi kelas target
data['PlacementStatus'].value_counts()

data['PlacementStatus'].value_counts(normalize=True) * 100

In [ ]:
#visualisasi distribbusi target
plt.figure(figsize=(7,5))

sns.countplot(data=data, x='PlacementStatus')

plt.title('Distribusi Status Penempatan Kerja')
plt.xlabel('Placement Status')
plt.ylabel('Jumlah Mahasiswa')

plt.show()

#visualisasi distribusi  variabel numerik
data.select_dtypes(include=['int64', 'float64']).drop(
    columns=['StudentID']
).hist(figsize=(15,10), bins=15)

plt.suptitle('Distribusi Variabel Numerik')
plt.tight_layout()
plt.show()

#visualisasi boxplot
plt.figure(figsize=(12,6))

sns.boxplot(data=numerik)

plt.title('Boxplot Variabel Numerik')
plt.xticks(rotation=45)

plt.show()

#visualisasi heatmap korelasi
plt.figure(figsize=(12,8))

sns.heatmap(
    data.select_dtypes(include=['int64', 'float64']).drop(
        columns=['StudentID']
    ).corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Heatmap Korelasi Antarvariabel Numerik')

plt.show()

**2. Validasi Data**

In [ ]:
#Validasi kualitas data
#Memeriksa missing value
data.isnull().sum()

In [ ]:
#Memeriksa data duplikat
data.duplicated().sum()

In [ ]:
#Memeriksa duplikasi ID
data['StudentID'].duplicated().sum()

In [ ]:
#Memeriksa konsistensi kategori
for kolom in ['ExtracurricularActivities',
              'PlacementTraining',
              'PlacementStatus']:

    print(kolom)
    print(data[kolom].value_counts(dropna=False))
    print()

In [ ]:
#Memeriksa rentang nilai numerik
data.describe().T[['min', 'max']]

In [ ]:
#Memeriksa indikasi pencilan menggunakan IQR
# 'numerik' already excludes 'StudentID' from the previous step
for kolom in numerik.columns:

    Q1 = data[kolom].quantile(0.25)
    Q3 = data[kolom].quantile(0.75)

    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    pencilan = data[
        (data[kolom] < batas_bawah) |
        (data[kolom] > batas_atas)
    ]

    print(kolom, ":", len(pencilan), "indikasi pencilan")

In [ ]:
#Pembersihan data
data_clean = data.copy()

# Menghapus data duplikat
data_clean = data_clean.drop_duplicates()

# Memperbaiki tipe data kategorik
kolom_kategorik = [
    'ExtracurricularActivities',
    'PlacementTraining',
    'PlacementStatus'
]

for kolom in kolom_kategorik:
    data_clean[kolom] = data_clean[kolom].astype(str).str.strip()

# Memastikan jumlah data setelah pembersihan
print("Dimensi data awal:", data.shape)
print("Dimensi data setelah pembersihan:", data_clean.shape)

# Memeriksa kembali missing value
print("\nMissing Value:")
print(data_clean.isnull().sum())

# Memeriksa kembali duplikat
print("\nJumlah Duplikat:")
print(data_clean.duplicated().sum())